# Raster Resampling

The `resample` function changes a raster's resolution (cell size) without
changing its CRS. This is the operation you'd reach for when you need to
match two rasters to a common grid or reduce a raster's memory footprint
before analysis.

**Methods**:

| Method | Direction | Best for |
|--------|-----------|----------|
| `nearest` | up/down | Categorical data, fast preview |
| `bilinear` | up/down | Smooth continuous surfaces |
| `cubic` | up/down | High-quality continuous surfaces |
| `average` | down only | Aggregating high-res to low-res |
| `min`, `max` | down only | Extremes within each output cell |
| `median` | down only | Robust centre, ignores outliers |
| `mode` | down only | Majority class in categorical rasters |

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import resample
from xrspatial.terrain import generate_terrain

## Generate synthetic terrain

In [ ]:
dem = generate_terrain(width=200, height=200)
# Assign a regular coordinate grid
dem = dem.assign_coords(
    y=np.linspace(100, 0, dem.sizes['y']),
    x=np.linspace(0, 100, dem.sizes['x']),
)
dem.attrs['res'] = (0.5, 0.5)

fig, ax = plt.subplots(figsize=(6, 5))
dem.plot(ax=ax, cmap='terrain')
ax.set_title(f'Original DEM ({dem.shape[0]}x{dem.shape[1]}, res={dem.attrs["res"][0]:.1f}m)')
plt.tight_layout()

## Downsample with `scale_factor`

In [ ]:
down = resample(dem, scale_factor=0.25, method='bilinear')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
dem.plot(ax=axes[0], cmap='terrain')
axes[0].set_title(f'Original ({dem.shape[0]}x{dem.shape[1]})')
down.plot(ax=axes[1], cmap='terrain')
axes[1].set_title(f'Downsampled 4x ({down.shape[0]}x{down.shape[1]})')
plt.tight_layout()

## Upsample with `target_resolution`

In [ ]:
up = resample(down, target_resolution=0.5, method='cubic')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
down.plot(ax=axes[0], cmap='terrain')
axes[0].set_title(f'Coarse ({down.shape[0]}x{down.shape[1]})')
up.plot(ax=axes[1], cmap='terrain')
axes[1].set_title(f'Upsampled to 0.5m ({up.shape[0]}x{up.shape[1]})')
plt.tight_layout()

## Compare resampling methods

In [ ]:
methods = ['nearest', 'bilinear', 'cubic', 'average']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, method in zip(axes, methods):
    out = resample(dem, scale_factor=0.1, method=method)
    out.plot(ax=ax, cmap='terrain', add_colorbar=False)
    ax.set_title(method)
    ax.set_aspect('equal')

plt.suptitle('Downsample 10x with different methods', y=1.02)
plt.tight_layout()

## Categorical raster with `mode`

In [ ]:
from xrspatial import equal_interval

# Classify elevation into 5 zones
classes = equal_interval(dem, k=5)
classes.attrs = dem.attrs.copy()
classes = classes.assign_coords(dem.coords)

# Downsample: mode preserves class boundaries
classes_down = resample(classes.astype('float32'),
                        scale_factor=0.2, method='mode')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
classes.plot(ax=axes[0], cmap='Set2')
axes[0].set_title(f'Classes ({classes.shape[0]}x{classes.shape[1]})')
classes_down.plot(ax=axes[1], cmap='Set2')
axes[1].set_title(f'Mode downsample ({classes_down.shape[0]}x{classes_down.shape[1]})')
plt.tight_layout()

## Works with Dask

In [ ]:
import dask.array as da

dask_dem = dem.copy()
dask_dem.data = da.from_array(dem.values, chunks=(100, 100))

result = resample(dask_dem, scale_factor=0.5, method='bilinear')
print(f'Input:  {dask_dem.shape} (dask, chunks={dask_dem.data.chunksize})')
print(f'Output: {result.shape} (dask, chunks={result.data.chunksize})')
print(f'Computed shape: {result.compute().shape}')